# JRDB pseudo-detection robustness protocol

This lightweight notebook reads the saved TrackEval summaries; it does not run trackers or recompute metrics. The protocol contains one Clean condition plus three levels each of Detector dropout, Detection instability, and Combined perturbations. All conditions use the same canonical global JRDB ground truth. Cells 6–8 create three independent HOTA figures.

In [ ]:
# Cell 1 — Imports
from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
from IPython.display import display

try:
    import yaml
except ImportError:
    yaml = None

In [ ]:
# Cell 2 — Editable paths, tracker selection, and plotting constants
RESULTS_TEST_DIR = Path(
    os.environ.get(
        "JRDB_PSEUDO_RESULTS_DIR",
        "./outputs/results/test",
    )
).expanduser().resolve()
FIGURE_DIR = Path(
    os.environ.get(
        "JRDB_PSEUDO_FIGURE_DIR",
        "./figs_pseudo_detection",
    )
).expanduser().resolve()
TABLE_DIR = FIGURE_DIR / "tables"
SPEC_PATH = Path(
    os.environ.get(
        "JRDB_PSEUDO_SPEC_PATH",
        "./pseudo_det_spec.yaml",
    )
).expanduser().resolve()
_manifest_value = os.environ.get("JRDB_PSEUDO_MANIFEST", "").strip()
MANIFEST_PATH = (
    Path(_manifest_value).expanduser().resolve()
    if _manifest_value
    else None
)

GLOBAL_COORDS = True
REFERENCE_TRACKER_BASE = "pedreftrack_gt_assisted"
TRACKER_BASES = [
    "pedreftrack_no_gt",
    "fastpoly",
    "simpletrack",
    "elptnet",
    "gnnpmb",
    "cbmot",
    "ab3dmot",
]
REQUIRE_COMPLETE_TRACKERS = False

PSEUDO_VARIANTS = {
    "clean": ("clean", "Clean"),
    "dropout_L1": ("dropout", "L1"),
    "dropout_L2": ("dropout", "L2"),
    "dropout_L3": ("dropout", "L3"),
    "instability_L1": ("instability", "L1"),
    "instability_L2": ("instability", "L2"),
    "instability_L3": ("instability", "L3"),
    "combined_L1": ("combined", "L1"),
    "combined_L2": ("combined", "L2"),
    "combined_L3": ("combined", "L3"),
}
PLOT_MODE_ORDER = ["instability", "dropout", "combined"]
CONDITION_ORDER = ["Clean", "L1", "L2", "L3"]
PLOT_CLEAN = False
PLOT_CONDITION_ORDER = (
    CONDITION_ORDER
    if PLOT_CLEAN
    else ["L1", "L2", "L3"]
)
CONDITION_LABELS = {
    "Clean": "Clean",
    "L1": "Mild",
    "L2": "Moderate",
    "L3": "Severe",
}
MODE_CAPTIONS = {
    "instability": "Detection instability",
    "dropout": "Detector dropout",
    "combined": "Combined perturbations",
}

TRACKER_STYLES = {
    "pedreftrack_gt_assisted": {"label": "GT-assisted PedRefTrack", "color": "#054D6B", "linestyle": "-"},
    "pedreftrack_no_gt": {"label": "PedRefTrack (no GT)", "color": "#38A4DC", "linestyle": "-"},
    "fastpoly": {"label": "Fast-Poly", "color": "#F4A641", "linestyle": "-"},
    "simpletrack": {"label": "SimpleTrack", "color": "#009E73", "linestyle": "-"},
    "elptnet": {"label": "ELPTNet (box)", "color": "#D55E00", "linestyle": "-"},
    "gnnpmb": {"label": "GNN-PMB", "color": "#CC79A7", "linestyle": "-"},
    "cbmot": {"label": "CBMOT", "color": "#7A5195", "linestyle": "-"},
    "ab3dmot": {"label": "AB3DMOT", "color": "#879AA3", "linestyle": "-"},
}
COMMON_MARKER = "o"
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = False
PAGE_WIDTH_IN = 7.0
SCAI = {
    "bg": "#FFFFFF",
    "ax_bg": "#FAFBFC",
    "spine": "#D0D0D0",
    "spine_dark": "#333333",
    "text": "#111111",
    "control": "#1A1A1A",
}

In [ ]:
# Cell 3 — Loading, validation, and table helpers
_variant_expression = "|".join(
    re.escape(name) for name in sorted(PSEUDO_VARIANTS, key=len, reverse=True)
)
VARIANT_PATTERN = re.compile(
    rf"^(?P<tracker>.+?)__global"
    rf"(?P<mode>_(?:no_gt|gt_assisted))?_(?P<variant>{_variant_expression})$"
)

def tracker_label(tracker):
    return TRACKER_STYLES.get(tracker, {"label": tracker.replace("_", " ")})["label"]

def tracker_color(tracker):
    if tracker not in TRACKER_STYLES:
        raise KeyError(f"No stable color assigned to tracker: {tracker}")
    return TRACKER_STYLES[tracker]["color"]

def tracker_linestyle(tracker):
    return TRACKER_STYLES.get(tracker, {"linestyle": "-"})["linestyle"]

def parse_result_directory(name):
    match = VARIANT_PATTERN.fullmatch(name)
    if match is None:
        return None
    parsed = match.groupdict()
    if not GLOBAL_COORDS:
        return None
    tracker = parsed["tracker"] + (parsed["mode"] or "")
    failure_mode, level = PSEUDO_VARIANTS[parsed["variant"]]
    return {
        "tracker": tracker,
        "variant": parsed["variant"],
        "failure_mode": failure_mode,
        "level": level,
    }

def load_combined_metrics(path):
    frame = pd.read_csv(path)
    frame.columns = [str(column).strip() for column in frame.columns]
    if "HOTA" not in frame.columns:
        raise KeyError(f"{path} lacks HOTA")
    if "seq" in frame.columns:
        combined = frame[frame["seq"].astype(str).str.strip().str.upper() == "COMBINED"]
    else:
        combined = pd.DataFrame()
    if combined.empty:
        warnings.warn(f"{path}: COMBINED row absent; using final row")
        combined = frame.tail(1)
    row = combined.iloc[0]
    values = {}
    for column in frame.columns:
        if column == "seq":
            continue
        numeric = pd.to_numeric(pd.Series([row[column]]), errors="coerce").iloc[0]
        if pd.notna(numeric):
            values[column] = float(numeric)
    if not 0.0 <= values["HOTA"] <= 100.0:
        raise ValueError(f"{path}: HOTA is not a percentage in [0, 100]")
    return values

def discover_results(root):
    if not root.is_dir():
        raise FileNotFoundError(f"Pseudo-result root not found: {root}")
    rows = []
    for directory in sorted(root.iterdir()):
        if not directory.is_dir():
            continue
        parsed = parse_result_directory(directory.name)
        if parsed is None:
            continue
        summary = directory / "pedestrian_summary.csv"
        rows.append({**parsed, "result_name": directory.name, "summary_csv": summary, "summary_exists": summary.is_file()})
    return pd.DataFrame(rows)

def completeness_table(discovered, trackers):
    expected = set(PSEUDO_VARIANTS)
    rows = []
    for tracker in trackers:
        subset = discovered[(discovered["tracker"] == tracker) & discovered["summary_exists"]]
        have = set(subset["variant"])
        missing = [name for name in PSEUDO_VARIANTS if name not in have]
        rows.append({"tracker": tracker, "label": tracker_label(tracker), "found": len(have & expected), "expected": len(expected), "complete": not missing, "missing": ", ".join(missing)})
    return pd.DataFrame(rows)

def load_metric_rows(discovered, trackers):
    selected = discovered[discovered["tracker"].isin(trackers) & discovered["summary_exists"]].copy()
    loaded = selected["summary_csv"].map(load_combined_metrics)
    return pd.concat([selected, pd.DataFrame(loaded.tolist(), index=selected.index)], axis=1)

def expand_clean_for_each_mode(raw):
    nonclean = raw[raw["failure_mode"] != "clean"].copy()
    clean = raw[raw["failure_mode"] == "clean"].copy()
    clean_copies = []
    for mode in PLOT_MODE_ORDER:
        copy = clean.copy()
        copy["failure_mode"] = mode
        clean_copies.append(copy)
    expanded = pd.concat([nonclean] + clean_copies, ignore_index=True)
    expanded["condition_index"] = expanded["level"].map({name: index for index, name in enumerate(CONDITION_ORDER)})
    return expanded

def add_condition_reference(frame):
    reference = frame[frame["tracker"] == REFERENCE_TRACKER_BASE][["failure_mode", "level", "HOTA"]].drop_duplicates(["failure_mode", "level"])
    expected = len(PLOT_MODE_ORDER) * len(CONDITION_ORDER)
    if len(reference) != expected:
        raise RuntimeError(f"Reference requires {expected} plot conditions; found {len(reference)}")
    reference = reference.rename(columns={"HOTA": "reference_HOTA"})
    merged = frame.merge(reference, on=["failure_mode", "level"], how="left", validate="many_to_one")
    merged["delta_HOTA"] = merged["HOTA"] - merged["reference_HOTA"]
    return merged

def load_spec(path):
    if not path.is_file():
        warnings.warn(f"Spec unavailable: {path}; parameter audit skipped")
        return None
    if yaml is None:
        raise ImportError("Install PyYAML to inspect the specification")
    with path.open("r", encoding="utf-8") as stream:
        return yaml.safe_load(stream)

def variant_parameter_table(spec):
    if spec is None:
        return pd.DataFrame()
    rows = [{"variant": "clean", "mode": "clean", "severity": "Clean", "parameters": "No injected degradation"}]
    sweeps = spec.get("single_mode_sweeps", {})
    for mode in ("dropout", "instability"):
        for item in sweeps.get(mode, []):
            variant = f"{mode}_{item['name']}"
            if variant not in PSEUDO_VARIANTS:
                continue
            level = PSEUDO_VARIANTS[variant][1]
            parameters = ", ".join(f"{key}={value}" for key, value in item.items() if key != "name")
            rows.append({"variant": variant, "mode": mode, "severity": CONDITION_LABELS[level], "parameters": parameters})
    for item in spec.get("combos", []):
        variant = str(item.get("name", ""))
        if variant not in PSEUDO_VARIANTS:
            continue
        level = PSEUDO_VARIANTS[variant][1]
        use = item.get("use", {})
        parameters = ", ".join(f"{key}={value}" for key, value in use.items())
        rows.append({"variant": variant, "mode": "combined", "severity": CONDITION_LABELS[level], "parameters": parameters})
    order = {name: index for index, name in enumerate(PSEUDO_VARIANTS)}
    table = pd.DataFrame(rows)
    table["order"] = table["variant"].map(order)
    return table.sort_values("order").drop(columns="order").reset_index(drop=True)

def load_manifest_gt_map(path):
    if path is None or not path.is_file():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    rows = []
    for item in manifest.get("variants", []):
        name = str(item.get("name", ""))
        if name in PSEUDO_VARIANTS:
            rows.append({"variant": name, "labels_subdir": item.get("labels_subdir"), "gt_tracker_name_global": item.get("gt_tracker_name_global")})
    return pd.DataFrame(rows)

In [ ]:
# Cell 4 — Plotting functions
def apply_scai_white_style():
    mpl.rcParams.update({
        "figure.facecolor": SCAI["bg"], "axes.facecolor": SCAI["ax_bg"],
        "axes.edgecolor": SCAI["spine_dark"], "axes.labelcolor": SCAI["text"],
        "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": False, "xtick.color": SCAI["text"], "ytick.color": SCAI["text"],
        "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 9.0, "axes.labelsize": 9.0, "xtick.labelsize": 8.0, "ytick.labelsize": 8.0,
        "legend.fontsize": 8.0, "legend.frameon": True, "legend.facecolor": SCAI["ax_bg"],
        "legend.edgecolor": SCAI["spine"], "lines.linewidth": 2.0, "lines.markersize": 5.0,
        "lines.solid_capstyle": "round", "savefig.dpi": 300,
        # "savefig.bbox": "tight",
        "savefig.facecolor": SCAI["bg"], "pdf.fonttype": 42, "svg.fonttype": "none",
    })

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SCAI["spine_dark"])
    ax.spines["bottom"].set_color(SCAI["spine_dark"])

def add_below_caption(ax, title, y=-0.28):
    ax.text(0.5, y, title, transform=ax.transAxes, ha="center", va="top", fontsize=9, color=SCAI["text"], clip_on=False)

def set_reference_strip_limits(ax, values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not values.size:
        return
    low, high = float(values.min()), float(values.max())
    span = high - low
    pad = max(0.10, 0.18 * max(span, 0.5))
    ax.set_ylim(low - pad, high + pad)
    ax.set_yticks([low] if np.isclose(low, high) else [low, high])
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.1f}"))
    ax.minorticks_off()

def set_delta_limits(ax, values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not values.size:
        ax.set_ylim(-1, 1)
        return
    low, high = min(0.0, float(values.min())), max(0.0, float(values.max()))
    pad = 0.08 * max(high - low, 0.5)
    ax.set_ylim(low - pad, high + pad)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

def legend_handles(trackers):
    ordered = [REFERENCE_TRACKER_BASE] + list(trackers)
    return [Line2D([0], [0], color=tracker_color(tracker), linestyle=tracker_linestyle(tracker), marker=COMMON_MARKER, label=tracker_label(tracker)) for tracker in ordered]

def plot_hota_mode(frame, trackers, mode):
    apply_scai_white_style()
    fig = plt.figure(figsize=(PAGE_WIDTH_IN / 2.0, PAGE_WIDTH_IN / 2.5), dpi=300)
    grid = fig.add_gridspec(2, 1, height_ratios=[0.6, 2.10], hspace=0.10)
    ax_reference = fig.add_subplot(grid[0, 0])
    ax_delta = fig.add_subplot(grid[1, 0], sharex=ax_reference)
    x = np.arange(len(PLOT_CONDITION_ORDER), dtype=float)
    mode_frame = frame[frame["failure_mode"] == mode]
    reference = mode_frame[mode_frame["tracker"] == REFERENCE_TRACKER_BASE].set_index("level").reindex(PLOT_CONDITION_ORDER)
    reference_values = reference["HOTA"].to_numpy(float)
    ax_reference.plot(x, reference_values, color=tracker_color(REFERENCE_TRACKER_BASE), linestyle=tracker_linestyle(REFERENCE_TRACKER_BASE), marker=COMMON_MARKER)
    ax_reference.set_ylabel("HOTA (%)", fontsize=7.5, labelpad=4)
    ax_reference.tick_params(axis="x", bottom=False, labelbottom=False)
    set_reference_strip_limits(ax_reference, reference_values)
    despine(ax_reference)
    delta_values = []
    for tracker in trackers:
        tracker_frame = mode_frame[mode_frame["tracker"] == tracker].set_index("level").reindex(PLOT_CONDITION_ORDER)
        values = tracker_frame["delta_HOTA"].to_numpy(float)
        delta_values.extend(values.tolist())
        ax_delta.plot(x, values, color=tracker_color(tracker), linestyle=tracker_linestyle(tracker), marker=COMMON_MARKER, label=tracker_label(tracker))
    ax_delta.axhline(0.0, color=SCAI["control"], linestyle="--", linewidth=1.0, zorder=0)
    ax_delta.set_xticks(x, [CONDITION_LABELS[name] for name in PLOT_CONDITION_ORDER])
    ax_delta.set_xlabel("Corruption severity")
    ax_delta.set_ylabel(r"$\Delta$HOTA (pp)")
    set_delta_limits(ax_delta, delta_values)
    despine(ax_delta)
    # add_below_caption(ax_delta, MODE_CAPTIONS[mode])
    fig.legend(handles=legend_handles(trackers), loc="upper center", bbox_to_anchor=(0.505, 0.99), ncol=3, columnspacing=0.5, handlelength=1.5, fontsize=7.5)
    fig.subplots_adjust(top=0.76, bottom=0.14, left=0.18, right=0.98)
    return fig

def save_and_display(fig, basename):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    if SAVE_PNG:
        fig.savefig(FIGURE_DIR / f"{basename}.png", dpi=300)
    if SAVE_PDF:
        fig.savefig(FIGURE_DIR / f"{basename}.pdf", dpi=300)
    if SAVE_SVG:
        fig.savefig(FIGURE_DIR / f"{basename}.svg", dpi=300)
    display(fig)
    plt.close(fig)

In [ ]:
# Cell 5 — Load the ten conditions and construct matched deltas
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
spec = load_spec(SPEC_PATH)
parameters = variant_parameter_table(spec)
manifest_gt_map = load_manifest_gt_map(MANIFEST_PATH)
discovered_results = discover_results(RESULTS_TEST_DIR)
selected_bases = [REFERENCE_TRACKER_BASE] + TRACKER_BASES
result_completeness = completeness_table(discovered_results, selected_bases)
missing_styles = [tracker for tracker in selected_bases if tracker not in TRACKER_STYLES]
if missing_styles:
    raise KeyError("Add TRACKER_STYLES entries for: " + ", ".join(missing_styles))
reference_status = result_completeness[result_completeness["tracker"] == REFERENCE_TRACKER_BASE].iloc[0]
if not bool(reference_status["complete"]):
    raise RuntimeError("GT-assisted PedRefTrack reference is incomplete: " + str(reference_status["missing"]))
incomplete_selected = result_completeness[(result_completeness["tracker"].isin(TRACKER_BASES)) & (~result_completeness["complete"])]
if REQUIRE_COMPLETE_TRACKERS and not incomplete_selected.empty:
    details = "\n".join(f"- {row.tracker}: {row.missing}" for row in incomplete_selected.itertuples(index=False))
    raise RuntimeError("Selected trackers are missing results:\n" + details)
ACTIVE_TRACKERS = list(TRACKER_BASES) if REQUIRE_COMPLETE_TRACKERS else [tracker for tracker in TRACKER_BASES if int(result_completeness.loc[result_completeness["tracker"] == tracker, "found"].iloc[0]) > 0]
raw_metrics = load_metric_rows(discovered_results, [REFERENCE_TRACKER_BASE] + ACTIVE_TRACKERS)
pseudo_hota = add_condition_reference(expand_clean_for_each_mode(raw_metrics))
pseudo_hota = pseudo_hota.sort_values(["failure_mode", "condition_index", "tracker"]).reset_index(drop=True)
display(parameters)
display(result_completeness)

In [ ]:
# Cell 6 — Detection-instability HOTA degradation
instability_figure = plot_hota_mode(pseudo_hota, ACTIVE_TRACKERS, "instability")
save_and_display(instability_figure, "pseudo_detection_instability_hota")

In [ ]:
# Cell 7 — Detector-dropout HOTA degradation
dropout_figure = plot_hota_mode(pseudo_hota, ACTIVE_TRACKERS, "dropout")
save_and_display(dropout_figure, "pseudo_detection_dropout_hota")

In [ ]:
# Cell 8 — Combined-perturbation HOTA degradation
combined_figure = plot_hota_mode(pseudo_hota, ACTIVE_TRACKERS, "combined")
save_and_display(combined_figure, "pseudo_detection_combined_hota")

In [ ]:
# Cell 9 — Numerical audits and full decomposition export
reference_hota = pseudo_hota[pseudo_hota["tracker"] == REFERENCE_TRACKER_BASE].pivot(index="failure_mode", columns="level", values="HOTA").reindex(index=PLOT_MODE_ORDER, columns=CONDITION_ORDER)
delta_hota = pseudo_hota[pseudo_hota["tracker"] != REFERENCE_TRACKER_BASE].pivot_table(index=["failure_mode", "level"], columns="tracker", values="delta_HOTA", aggfunc="first")
full_export = raw_metrics.copy()
full_export["summary_csv"] = full_export["summary_csv"].astype(str)
full_export.to_csv(TABLE_DIR / "pseudo_detection_full_metrics.csv", index=False)
pseudo_hota[["tracker", "failure_mode", "level", "HOTA", "reference_HOTA", "delta_HOTA"]].to_csv(TABLE_DIR / "pseudo_detection_hota_plot_data.csv", index=False)
print(f"Results directory: {RESULTS_TEST_DIR}")
print(f"Figure directory:  {FIGURE_DIR}")
print(f"Active comparison trackers: {ACTIVE_TRACKERS}")
print("All ten conditions use canonical global GT; confuser folders are ignored.")
print("ΔHOTA = tracker − condition-matched GT-assisted PedRefTrack")
print("GT-assisted PedRefTrack absolute HOTA:")
display(reference_hota.round(3))
print("ΔHOTA to GT-assisted PedRefTrack (pp):")
display(delta_hota.round(3))
print(f"Saved full decomposition table: {TABLE_DIR / 'pseudo_detection_full_metrics.csv'}")
if manifest_gt_map.empty:
    print("Manifest audit skipped. Set JRDB_PSEUDO_MANIFEST to inspect input provenance.")
else:
    missing = [name for name in PSEUDO_VARIANTS if name not in set(manifest_gt_map["variant"])]
    if missing:
        raise RuntimeError("Manifest lacks protocol variants: " + ", ".join(missing))
    display(manifest_gt_map.sort_values("variant").reset_index(drop=True))